# LLM harness A/B arena

Inspect ten diverse task contracts, prove the control/treatment boundary, run the dependency-free 20-trial transport smoke, and prepare (but do not execute) a real multi-harness/model matrix. The fixture calls no model and provides no efficacy evidence.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from solutiongraph.agent_bench import (
    REFERENCE_AGENT_TASKS,
    command_matrix_example_suite,
    iter_trial_plans,
    reference_agent_benchmark_suite,
    run_agent_benchmark,
    write_agent_benchmark_suite,
)
from solutiongraph.agent_bench.workspace import materialize_workspace

[(task.spec.id, task.spec.title, task.spec.template_id) for task in REFERENCE_AGENT_TASKS]

In [ ]:
task = REFERENCE_AGENT_TASKS[-1]
print(task.spec.mermaid())
print('public:', task.spec.public_case_ids)
print('sealed:', task.spec.sealed_case_ids)

In [ ]:
suite = reference_agent_benchmark_suite()
plans = tuple(iter_trial_plans(suite))
print('suite digest:', suite.digest)
print('planned trials:', len(plans))
print('first arm in each pair:', [plans[i].condition for i in range(0, len(plans), 2)])

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
pair_task = REFERENCE_AGENT_TASKS[0]
pair_plans = [plan for plan in plans if plan.task_id == pair_task.spec.id]
by_condition = {plan.condition: plan for plan in pair_plans}
with TemporaryDirectory() as temporary:
    root = Path(temporary)
    control = materialize_workspace(root / 'control', repo_root, pair_task, by_condition['control'])
    treatment = materialize_workspace(root / 'solutiongraph', repo_root, pair_task, by_condition['solutiongraph'])
    print('identical prompt:', control.prompt_digest == treatment.prompt_digest)
    print('control context bytes:', control.context_bytes)
    print('treatment context bytes:', treatment.context_bytes)
    print('control aggregate bytes:', (root / 'control' / 'context' / 'AGENT_CONTEXT.md').stat().st_size)
    print('treatment context files:', len(list((root / 'solutiongraph' / 'context').rglob('*'))))

## Execute the safe mechanism smoke

This uses the evaluator-owned deterministic fixture. A correct result accepts all 20 trials, finds practical equivalence between arms, and emits no winner or promotion.

In [ ]:
_arena_tmp = TemporaryDirectory()
result = run_agent_benchmark(suite, Path(_arena_tmp.name) / 'smoke')
result.to_dict()

In [ ]:
overall = [effect for effect in result.report.effects if effect.scope == 'overall']
for effect in overall:
    print(effect.metric, effect.pairs, effect.inference, (effect.confidence_lower, effect.confidence_upper))
print('decisions:', result.report.decisions)
print('offline report:', result.report_html)

## Prepare a real experiment without running it

The generated file keeps external OpenCode, Aider, private command harnesses, and non-fixture models disabled. Pin revisions, compatibility, settings, credentials by variable name, budgets, and isolation before enabling anything.

In [ ]:
config_dir = Path('.artifacts')
config_dir.mkdir(exist_ok=True)
command_suite = command_matrix_example_suite()
config_path = write_agent_benchmark_suite(command_suite, config_dir / 'agent-benchmark-command-matrix.json')
print(config_path)
print('currently enabled trials:', command_suite.total_trials)
print('Run solutiongraph agent-bench plan on this file after editing; execution requires --allow-external.')